In [1]:
!pip install pydantic langgraph


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [15]:
from pydantic import BaseModel
from typing import List, Dict
import pandas as pd
import ast

In [16]:
df = pd.read_csv("kyc_synthetic_data.csv")
df.head()

,ocr_text,name,matched_name,transactions,pep_flag,sanctions_flag
0,Name: Vihaan Verma\nDOB: 08/07/1985\nPAN: RRFP...,Vihaan Verma,Vihaan Verma,"[{'amount': 84254, 'type': 'credit', 'mode': '...",0,1
1,Name: Ishaan Gupta\nDOB: 08/03/1996\nPAN: JHRB...,Ishaan Gupta,Ishaan Gupta,"[{'amount': 14337, 'type': 'debit', 'mode': 'c...",1,0
2,Name: Saanvi Reddy\nDOB: 04/10/2002\nPAN: BXVQ...,Saanvi Reddy,Saanvi Reddy,"[{'amount': 94875, 'type': 'credit', 'mode': '...",0,0
3,Name: Saanvi Joshi\nDOB: 12/05/1985\nPAN: QEPT...,Saanvi Joshi,Saanvi Joshi,"[{'amount': 92720, 'type': 'debit', 'mode': 'c...",1,0
4,Name: Vihaan Sharma\nDOB: 10/10/1995\nPAN: NFU...,Vihaan Sharma,Vihaan Sharma,"[{'amount': 60123, 'type': 'debit', 'mode': 'c...",0,1


In [17]:
df["transactions"] = df["transactions"].apply(ast.literal_eval)

In [18]:
class IdentityOutput(BaseModel):
    match_score: float


class TransactionOutput(BaseModel):
    credit: float
    debit: float
    cash_ratio: float


class ComplianceOutput(BaseModel):
    risk: str


class RiskOutput(BaseModel):
    final_score: float
    risk_label: str

In [19]:
def ocr_agent(text: str):
    data = {"name": "", "dob": "", "id_number": ""}

    for line in text.split("\n"):
        if "Name" in line:
            data["name"] = line.split(":")[-1].strip()
        if "DOB" in line:
            data["dob"] = line.split(":")[-1].strip()
        if "PAN" in line:
            data["id_number"] = line.split(":")[-1].strip()

    return data

In [20]:
def identity_agent(name, matched_name):
    score = 0.98 if name.lower() == matched_name.lower() else 0.7
    return IdentityOutput(match_score=score)

In [21]:
def transaction_agent(transactions):

    credit = sum(t["amount"] for t in transactions if t["type"] == "credit")
    debit = sum(t["amount"] for t in transactions if t["type"] == "debit")

    cash_ratio = len([t for t in transactions if t["mode"] == "cash"]) / len(transactions)

    return TransactionOutput(
        credit=credit,
        debit=debit,
        cash_ratio=cash_ratio
    )

In [22]:
def compliance_agent(pep_flag, sanctions_flag):

    if sanctions_flag == 1:
        risk = "HIGH"
    elif pep_flag == 1:
        risk = "MEDIUM"
    else:
        risk = "LOW"

    return ComplianceOutput(risk=risk)

In [23]:
def risk_agent(identity, transaction, compliance):

    compliance_score = 0.2 if compliance.risk == "HIGH" else 0.7

    final_score = (
        0.3 * identity.match_score +
        0.4 * (1 - transaction.cash_ratio) +
        0.3 * compliance_score
    )

    # BUSINESS RULES (IMPORTANT)
    if compliance.risk == "HIGH":
        label = "HIGH"
    elif transaction.cash_ratio > 0.6:
        label = "HIGH"
    elif final_score > 0.7:
        label = "HIGH"
    elif final_score > 0.4:
        label = "MEDIUM"
    else:
        label = "LOW"

    return RiskOutput(
        final_score=final_score,
        risk_label=label
    )

In [12]:
ocr_text = """
Name: Aarav Mehta
DOB: 12/06/1994
PAN: ABCDE1234F
"""

transactions = [
    {"amount": 50000, "type": "credit", "mode": "salary"},
    {"amount": 20000, "type": "debit", "mode": "cash"}
]

In [13]:
ocr = ocr_agent(ocr_text)

identity = identity_agent(ocr["name"], "Aarav Mehta")

transaction = transaction_agent(transactions)

compliance = compliance_agent(0, 0)

risk = risk_agent(identity, transaction, compliance)

print("OCR:", ocr)
print("IDENTITY:", identity.model_dump())
print("TRANSACTIONS:", transaction.model_dump())
print("COMPLIANCE:", compliance.model_dump())
print("RISK:", risk.model_dump())

OCR: {'name': 'Aarav Mehta', 'dob': '12/06/1994', 'id_number': 'ABCDE1234F'}
IDENTITY: {'match_score': 0.98}
TRANSACTIONS: {'credit': 50000.0, 'debit': 20000.0, 'cash_ratio': 0.5}
COMPLIANCE: {'risk': 'LOW'}
RISK: {'final_score': 0.704, 'risk_label': 'HIGH'}


In [24]:
results = []

for _, row in df.iterrows():

    ocr = ocr_agent(row["ocr_text"])
    identity = identity_agent(row["name"], row["matched_name"])
    transaction = transaction_agent(row["transactions"])
    compliance = compliance_agent(row["pep_flag"], row["sanctions_flag"])
    risk = risk_agent(identity, transaction, compliance)

    results.append({
        "name": row["name"],
        "risk_label": risk.risk_label,
        "risk_score": risk.final_score
    })

results[:5]

[{'name': 'Vihaan Verma', 'risk_label': 'HIGH', 'risk_score': 0.554},
 {'name': 'Ishaan Gupta', 'risk_label': 'HIGH', 'risk_score': 0.604},
 {'name': 'Saanvi Reddy',
  'risk_label': 'HIGH',
  'risk_score': 0.7706666666666666},
 {'name': 'Saanvi Joshi',
  'risk_label': 'HIGH',
  'risk_score': 0.9039999999999999},
 {'name': 'Vihaan Sharma',
  'risk_label': 'HIGH',
  'risk_score': 0.6206666666666667}]